# Run the HydroCNHS FE Walter benchmark / 執行 FE Walter 規則基準

## Goal / 目標
Use the flood-control branch. Statistics are derived from frozen 1996–2015 inflow, before the 2016 sample. This statistical reference period is not a new calibration. The benchmark selects requested release; this repository runs hydrocnhs.Model.run, the Dam API, common physics and routing for all three methods.

使用 flood-control 分支；統計由固定 1996–2015 入流取得，早於 2016 範例，不是重新校正。Benchmark 只選請求放流；三方法皆由本 repo 執行 hydrocnhs.Model.run、Dam API、共同物理與 routing。

## Setup / 準備
Python 3.12 with repository dependencies. Local files only; no provider calls. / 使用 Python 3.12 與既有依賴；使用局部檔案，不呼叫 provider。

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
from tutorials.inspect_decision_flow import show_decision_flow
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: HydroCNHS-Delaware-Operations


## Steps

### Read statistics and coefficient rules / 讀取統計與年度係數

In [2]:
stats=json.loads((ROOT/'data/benchmark/fe_walter_statistics.json').read_text())
print({k:stats[k] for k in ['reference_start','reference_end','reference_day_count','mean_inflow_cms']})
print('gamma = capacity / (mean inflow * 86400 * 365.25)')
print('k = starting storage / (alpha * capacity); update on operating-year start')
print('Environment floor = 10% of historical mean inflow')

{'reference_start': '1996-01-01', 'reference_end': '2015-12-31', 'reference_day_count': 7305, 'mean_inflow_cms': 18.25446484681121}
gamma = capacity / (mean inflow * 86400 * 365.25)
k = starting storage / (alpha * capacity); update on operating-year start
Environment floor = 10% of historical mean inflow


### Run or verify three days / 執行或驗證三日

In [3]:
settings, verified=ensure_offline('examples/fe_walter/benchmark/experiment.hydroopslm.offline.yaml')
print(json.dumps(verified,indent=2))

{
  "status": "PASS",
  "days": 3,
  "reservoir_count": 1,
  "reservoir_days": 3,
  "date_start": "2016-01-01",
  "date_end": "2016-01-03",
  "audit_counts": {
    "PASS": 3
  },
  "max_reservoir_balance_residual_m3": 2.3283064365386963e-10,
  "max_system_balance_residual_m3": 2.3283064365386963e-10,
  "max_routing_difference_cms": 0.0,
  "final_channel_storage_m3": 0.0,
  "observed_network_calls": 0,
  "operating_strategy_accepted": null,
  "audit_binding_counts": {
    "versioned_suite": 3,
    "legacy_unbound": 0
  },
  "external_audit_pins_checked": []
}


## Checks / 檢查
Review printed inputs, counts, sources and residuals. Synthetic replies validate interfaces only. / 核對輸入、計數、來源與殘差；合成回覆僅驗證介面。

## Next steps / 下一步
This implementation matches the pinned Anik flood-control function across 162 differential vectors. It is a benchmark adaptation, not an official FE Walter operating rule.

此實作已在 162 組差分向量與固定 Anik flood-control 函數比對；這是 benchmark 調整版，不是 FE Walter 官方操作規則。